# GNAS

Cohort: `GNAS`, Phenopacket Store v0.1.27 (n = 103).

- Pseudohypoparathyroidism Ia (PHP-Ia), [OMIM:103580](https://omim.org/entry/103580), n = 80
- Pseudopseudohypoparathyroidism (PPHP), [OMIM:612463](https://omim.org/entry/612463), n = 23

Both subtypes share the Albright hereditary osteodystrophy (AHO) features and are
separated by parent of origin: maternally inherited inactivating GNAS variants cause
PHP-Ia with multi-hormone resistance, paternally inherited variants cause PPHP with AHO
alone. Disease membership is therefore a clinically meaningful synergy target.

All individuals derive from a single published series (PMID 25802881, Thiele et al.
2015, Mol Genet Genomic Med 3:111-120).

In [ ]:
from pathlib import Path

import pandas as pd
from ppktstore.registry import configure_phenopacket_registry

from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)

OUTDIR = Path("../case_studies/GNAS")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release="0.1.27") as ps:
    phenopackets = list(ps.iter_cohort_phenopackets("GNAS"))

print(f"Loaded {len(phenopackets)} phenopackets")

In [ ]:
# has_disease works on the phenopacket directly, no GPSEA cohort needed here
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

In [ ]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df

## Pairwise correlation

In [ ]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

t = results.results_table
print(f"tested pairs: {len(t)} | significant: {int((t['adj_p_value'] < 0.05).sum())}")
t.head()

In [ ]:
results.save_correlation_results(
    output_file=str(OUTDIR / "correlation_results_GNAS.csv")
)

In [ ]:
sig = results.results_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[["HPO_A_label", "HPO_B_label", "correlation", "adj_p_value", "n_individuals", "n_pmids"]]

In [ ]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort GNAS",
)

## Synergy: PHP-Ia vs PPHP

In [ ]:
condition = dataset.get_condition(
    has_disease("OMIM:103580"),
    name="PHP-Ia vs PPHP",
)

condition.value_counts(dropna=False)

In [ ]:
# lowered from the default of 30, the cohort splits 80/23
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=25,
)
synergy_results = synergy_analyzer.compute_synergy_matrix(condition=condition)

st = synergy_results.results_table
print(
    f"rows={len(st)} | synergy {st['synergy'].min():.4f} to {st['synergy'].max():.4f}"
    f" | positive={(st['synergy'] > 0).sum()} negative={(st['synergy'] < 0).sum()}"
)
st.head()

In [ ]:
# save_synergy_results() drops everything below synergy >= 0.01,
# so keep the unfiltered table for the dissynergy side
synergy_results.results_table.to_csv(
    OUTDIR / "synergy_full_GNAS_OMIM_103580.csv", index=False
)
synergy_results.save_synergy_results(
    output_file=str(OUTDIR / "synergy_matrix_GNAS_OMIM_103580.csv")
)

In [ ]:
synergy_results.plot_synergy_heatmap()

In [ ]:
COLS = ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals", "n_pmids"]

print("--- top synergy ---")
display(st.nlargest(10, "synergy")[COLS])
print("--- top dissynergy ---")
display(st.nsmallest(10, "synergy")[COLS])